# KBO 2022~2026 데이터 전처리와 2026 가을야구 예측

목적은 KBO 정규시즌 데이터만 사용해 2026년 현재 시점에서 가을야구 진출 가능성이 높은 팀을 예측하는 것입니다.

- 공식 데이터 출처: https://www.koreabaseball.com
- 분석 데이터 경로: `C:/Users/Admin/Documents/GitHub/Ml_Baseball/data`
- 예측 기준: 2026년 로컬 `팀_일자별순위.csv`의 최신 날짜
- 타깃 라벨: 최종 순위 5위 이내 여부

중요한 점은 누수 방지입니다. 훈련에는 “당일 일자별 순위 스냅샷”과 “전년도 팀/선수 전력 지표”만 넣고, 해당 시즌 최종 팀기록은 타깃 라벨 생성 외에는 쓰지 않습니다.

## 데이터로 가능한 인사이트와 추가 크롤링 필요성

현재 있는 2022~2026 CSV만으로도 가을야구 예측을 위한 기본 분석은 가능합니다.

가능한 분석:
- 4월 순위와 최종 5강의 관계
- 현재 승률, 게임차, 최근 10경기 흐름 기반의 실시간 예측
- 전년도 팀 타격/투수/수비/주루 지표가 다음 시즌 성적에 주는 신호
- 전년도 핵심 타자/투수 상위권 집계와 다음 시즌 5강 여부의 관계

추가 크롤링이 필요한 분석:
- 선수 프로필/통산/일자별/경기별/상황별 기록: 선수별 컨디션, 출전 지속성, 특정 상황 강점 분석
- 선수 이동 현황: 2026년 전력 변화 보정
- 팀 세부기록: OPS, 장타/출루, 투수 세부지표를 통한 설명력 강화

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\Admin\Documents\Codex\2026-04-28\files-mentioned-by-the-user-data")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import kbo_postseason_pipeline as pipe

DATA_DIR = Path(r"C:\Users\Admin\Documents\GitHub\Ml_Baseball\data")
OUT_DIR = Path(r"C:\Users\Admin\Documents\Codex\2026-04-28\files-mentioned-by-the-user-data\kbo_outputs")

artifacts = pipe.run_pipeline(DATA_DIR, OUT_DIR)
artifacts["summary"]

{'data_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\data',
 'out_dir': 'C:\\Users\\Admin\\Documents\\Codex\\2026-04-28\\files-mentioned-by-the-user-data\\kbo_outputs',
 'training_rows': 4629,
 'model_rows_total': 5190,
 'latest_2026_date': '2026-04-28',
 'predicted_top5': ['KT', 'LG', 'SSG', '삼성', '한화'],
 'april_current_top5_overlap_mean': 3.0,
 'validation_april_ensemble_top5_overlap_mean': 3.0,
 'validation_april_baseline_top5_overlap_mean': 3.0}

## 전처리 과정

파이프라인에서 수행하는 핵심 전처리입니다.

- `YYYYMMDD` 형태 날짜를 datetime으로 변환
- `최근10경기` 문자열에서 승/무/패와 최근 승률 추출
- `홈`, `방문` 문자열에서 홈/원정 승률 추출
- `IP`의 `5 1/3`, `5.1` 같은 이닝 표기를 소수 이닝으로 변환
- 팀별 최종 순위 5위 이내를 `postseason` 라벨로 생성
- 2023~2026 각 시즌에 대해 직전 시즌 팀/선수 피처를 붙임
- 2026은 라벨 없이 최신 스냅샷만 예측 대상으로 사용

In [2]:
model_df = artifacts["model_dataset"]
model_df[[
    "season", "date", "team", "rank", "games", "win_rate", "games_behind",
    "recent10_win_rate", "home_win_rate", "away_win_rate",
    "prev_final_rank", "prev_avg", "prev_era", "prev_whip", "postseason"
]].head(12)

,season,date,team,rank,games,win_rate,games_behind,recent10_win_rate,home_win_rate,away_win_rate,prev_final_rank,prev_avg,prev_era,prev_whip,postseason
0,2023,2023-04-01,SSG,1.0,1.0,1.0,0.0,1.0,1.0,NaN,1.0,0.254,3.87,1.29,1.0
1,2023,2023-04-01,키움,1.0,1.0,1.0,0.0,1.0,1.0,NaN,2.0,0.252,3.79,1.34,0.0
2,2023,2023-04-01,KT,1.0,1.0,1.0,0.0,1.0,1.0,NaN,4.0,0.254,3.51,1.25,1.0
3,2023,2023-04-01,NC,1.0,1.0,1.0,0.0,1.0,NaN,1.0,6.0,0.257,3.90,1.36,1.0
4,2023,2023-04-01,두산,1.0,1.0,1.0,0.0,1.0,1.0,NaN,9.0,0.255,4.45,1.48,1.0
5,2023,2023-04-01,LG,6.0,1.0,0.0,1.0,0.0,NaN,0.0,3.0,0.269,3.33,1.27,1.0
6,2023,2023-04-01,KIA,6.0,1.0,0.0,1.0,0.0,NaN,0.0,5.0,0.272,4.20,1.42,0.0
7,2023,2023-04-01,삼성,6.0,1.0,0.0,1.0,0.0,0.0,NaN,7.0,0.270,4.29,1.44,0.0
8,2023,2023-04-01,롯데,6.0,1.0,0.0,1.0,0.0,NaN,0.0,8.0,0.267,4.45,1.46,0.0
9,2023,2023-04-01,한화,6.0,1.0,0.0,1.0,0.0,NaN,0.0,10.0,0.245,4.83,1.51,0.0


## 2026 가을야구 예측 결과

`모델확률`은 과거 일자별 순위와 전년도 전력 지표로 학습한 로지스틱 모델의 출력입니다.
`스탠딩확률`은 현재 순위, 승률, 게임차, 최근 10경기를 반영한 안정화 점수입니다.
최종 `가을야구확률`은 두 값을 섞은 앙상블이며, 현재처럼 4월 데이터만 있는 상황에서 과적합을 줄이기 위한 장치입니다.

In [3]:
pred = pd.read_csv(OUT_DIR / "2026_postseason_predictions.csv", encoding="utf-8-sig")
pred[[
    "team", "rank", "games", "wins", "losses", "win_rate", "games_behind",
    "model_probability_pct", "standing_probability_pct",
    "postseason_probability_pct", "prediction_label"
]]

,team,rank,games,wins,losses,win_rate,games_behind,model_probability_pct,standing_probability_pct,postseason_probability_pct,prediction_label
0,KT,1.0,26.0,18.0,8.0,0.692,0.0,96.6,88.4,92.1,진출
1,LG,2.0,25.0,16.0,9.0,0.640,1.5,95.1,77.9,85.6,진출
2,SSG,3.0,25.0,15.0,10.0,0.600,2.5,94.5,70.9,81.5,진출
3,삼성,4.0,25.0,13.0,11.0,0.542,4.0,88.8,57.3,71.4,진출
4,한화,7.0,25.0,11.0,14.0,0.440,6.5,91.3,36.9,61.4,진출
5,NC,5.0,25.0,12.0,13.0,0.480,5.5,59.9,49.6,54.2,미진출
6,KIA,5.0,26.0,12.0,13.0,0.480,5.5,56.0,47.9,51.6,미진출
7,두산,8.0,26.0,10.0,15.0,0.400,7.5,53.4,28.9,39.9,미진출
8,키움,9.0,26.0,10.0,16.0,0.385,8.0,1.7,23.7,13.8,미진출
9,롯데,10.0,25.0,8.0,16.0,0.333,9.0,4.0,11.5,8.2,미진출


## 4월 순위만으로 어디까지 맞나

2023~2025년의 4월 마지막 스냅샷을 기준으로 보면, 당시 5위권 중 최종 5강에 남은 팀은 매년 3팀이었습니다.
즉 4월 순위는 의미 있는 신호지만 그대로 확정값처럼 쓰기에는 이릅니다.

In [4]:
pd.read_csv(OUT_DIR / "april_rank_insight.csv", encoding="utf-8-sig")

,season,april_latest_date,current_top5_overlap,current_top5_overlap_rate,rank_final_spearman
0,2023,2023-04-30,3,0.6,0.413376
1,2024,2024-04-30,3,0.6,0.418182
2,2025,2025-04-30,3,0.6,0.684848


## 시즌 홀드아웃 검증

한 시즌을 테스트로 빼고 나머지 시즌으로 학습했습니다. 표본 시즌이 2023~2025 세 시즌뿐이므로, 이 수치는 모델의 방향성을 보는 용도입니다.

In [5]:
pd.read_csv(OUT_DIR / "validation_leave_one_season.csv", encoding="utf-8-sig")

,test_season,slice,rows,team_accuracy,top5_overlap,baseline_current_top5_overlap
0,2023,daily_all_after_10g,1566,0.810983,NaN,NaN
1,2023,april_latest,10,0.600000,3.0,3.0
2,2023,season_latest,10,0.800000,4.0,5.0
3,2024,daily_all_after_10g,1530,0.696732,NaN,NaN
4,2024,april_latest,10,0.500000,3.0,3.0
5,2024,season_latest,10,0.800000,4.0,5.0
6,2025,daily_all_after_10g,1533,0.613829,NaN,NaN
7,2025,april_latest,10,0.700000,3.0,3.0
8,2025,season_latest,10,0.600000,5.0,5.0


## 모델이 민감하게 본 피처

표본 시즌이 적어서 계수의 부호를 인과로 해석하면 안 됩니다. 그래도 어떤 신호에 모델이 민감했는지 점검하는 용도로 사용합니다.

In [6]:
pd.read_csv(OUT_DIR / "feature_importance_coefficients.csv", encoding="utf-8-sig").head(15)

,feature,feature_ko,coefficient,abs_coefficient,direction
0,prev_sb_per_game,전년도 도루/G,0.731674,0.731674,positive
1,prev_postseason,전년도 가을야구,-0.641607,0.641607,negative
2,prev_hr_per_game,전년도 홈런/G,0.567073,0.567073,positive
3,rank,현재 순위,-0.548695,0.548695,negative
4,games_behind,현재 게임차,-0.539483,0.539483,negative
5,prev_errors_per_game,전년도 실책/G,-0.432260,0.432260,negative
6,prev_avg,전년도 팀 타율,-0.422841,0.422841,negative
7,prev_so_per9,전년도 K/9,-0.323826,0.323826,negative
8,prev_bb_per9,전년도 BB/9,-0.322600,0.322600,negative
9,prev_top5_pitcher_so,전년도 상위5투수 SO,0.293879,0.293879,positive


## 2026 팀 스냅샷

아래 테이블은 최신 2026 팀 순위와 현재까지의 팀 타자/투수/수비/주루 기본기록을 붙인 분석용 테이블입니다.
예측 모델에는 누수 문제 때문에 같은 시즌 최종 팀기록을 훈련 피처로 쓰지 않았지만, 현재 시즌 상태 설명에는 유용합니다.

In [7]:
snapshot = pd.read_csv(OUT_DIR / "2026_team_snapshot.csv", encoding="utf-8-sig")
snapshot[[
    "team", "rank", "games", "win_rate", "games_behind",
    "current_runs_per_game", "current_hr_per_game",
    "current_era", "current_whip", "current_errors_per_game", "current_sb_rate"
]].sort_values("rank")

,team,rank,games,win_rate,games_behind,current_runs_per_game,current_hr_per_game,current_era,current_whip,current_errors_per_game,current_sb_rate
0,KT,1.0,26.0,0.692,0.0,6.076923,0.846154,3.83,1.38,0.730769,61.9
1,LG,2.0,25.0,0.640,1.5,4.600000,0.560000,3.54,1.34,0.560000,75.0
2,SSG,3.0,25.0,0.600,2.5,5.400000,1.080000,4.24,1.48,0.880000,81.5
3,삼성,4.0,25.0,0.542,4.0,5.040000,0.720000,4.22,1.45,0.480000,71.4
4,NC,5.0,25.0,0.480,5.5,4.640000,0.960000,4.17,1.30,0.640000,82.2
5,KIA,5.0,26.0,0.480,5.5,4.846154,0.884615,4.54,1.39,0.615385,78.6
6,한화,7.0,25.0,0.440,6.5,5.800000,0.840000,5.24,1.67,1.040000,68.2
7,두산,8.0,26.0,0.400,7.5,4.384615,0.769231,4.52,1.52,0.653846,79.2
8,키움,9.0,26.0,0.385,8.0,3.461538,0.461538,4.45,1.56,0.923077,87.5
9,롯데,10.0,25.0,0.333,9.0,3.120000,0.800000,4.57,1.48,0.640000,80.0


## 실시간 크롤링 코드 실행

같은 폴더에 `kbo_realtime_crawler.py`를 생성했습니다. KBO 공식 사이트의 ASP.NET PostBack 구조를 따라 연도 선택, 페이지 이동, 팀 일자별 순위 이동을 처리합니다.

기본 실행:
```powershell
python kbo_realtime_crawler.py --year 2026 --data-dir "C:\Users\Admin\Documents\GitHub\Ml_Baseball\data"
```

터미널을 계속 켜두고 매일 00:00 KST에 실행:
```powershell
python kbo_realtime_crawler.py --year 2026 --watch-midnight
```

Windows 작업 스케줄러 등록:
```powershell
.\register_kbo_midnight_task.ps1 -PythonExe "python"
```

선수 프로필 전체 수집은 시간이 오래 걸릴 수 있으므로 필요할 때만:
```powershell
python kbo_realtime_crawler.py --year 2026 --profiles
```

In [8]:
from pathlib import Path

for path in [
    Path("kbo_postseason_pipeline.py"),
    Path("kbo_realtime_crawler.py"),
    Path("register_kbo_midnight_task.ps1"),
    OUT_DIR / "analysis_report.md",
    OUT_DIR / "2026_postseason_predictions.csv",
]:
    print(path.resolve())

C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\kbo_postseason_pipeline.py
C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\kbo_realtime_crawler.py
C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\register_kbo_midnight_task.ps1
C:\Users\Admin\Documents\Codex\2026-04-28\files-mentioned-by-the-user-data\kbo_outputs\analysis_report.md
C:\Users\Admin\Documents\Codex\2026-04-28\files-mentioned-by-the-user-data\kbo_outputs\2026_postseason_predictions.csv
